<a href="https://colab.research.google.com/github/Tecknique/200_ml/blob/main/FLA_UW_GUI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import io
import json
import re
import shutil
import threading
import zipfile
import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Optional

from IPython.display import HTML, display

# ----------------- config / deps -----------------
OWNER, REPO = "timrobinson", "UW-LFA-Analysis"
BRANCH = "main"
BASE_REL = Path("100microliters/Database")
IMG_EXTS = {".tif", ".tiff", ".jpg", ".jpeg"}
EXPORT_ROOT = Path("/content/roi_exports") if Path("/content").exists() else Path.cwd() / "roi_exports"
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

try:
    import cv2
    import numpy as np
    import requests
    import pandas as pd
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from scipy.signal import savgol_filter
except Exception:
    import sys
    import subprocess as sp

    sp.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "opencv-python-headless",
            "requests",
            "flask",
            "matplotlib",
            "pandas",
            "scipy",
        ],
        check=True,
    )
    import cv2  # noqa
    import numpy as np  # noqa
    import requests  # noqa
    import pandas as pd  # noqa
    import matplotlib  # noqa

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt  # noqa
    from scipy.signal import savgol_filter  # noqa


# ----------------- helpers -----------------
def _download_repo_zip(owner: str, repo: str, branch: str) -> bytes:
    def fetch(br: str) -> Optional[bytes]:
        url = f"https://github.com/{owner}/{repo}/archive/refs/heads/{br}.zip"
        r = requests.get(url, timeout=60)
        return r.content if r.status_code == 200 else None

    blob = fetch(branch) or (fetch("master") if branch == "main" else None)
    if blob is None:
        raise RuntimeError(f"Cannot download {owner}/{repo} ({branch}/master).")
    return blob


def _extract_zip_to_tmp(zip_bytes: bytes, base_dir: Path) -> Path:
    tmp_root = base_dir / "_uwlfa_tmp"
    if tmp_root.exists():
        shutil.rmtree(tmp_root)
    tmp_root.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(io.BytesIO(zip_bytes), "r") as zf:
        zf.extractall(tmp_root)
        top_dirs = sorted({Path(n).parts[0] for n in zf.namelist() if "/" in n})

    if not top_dirs:
        raise RuntimeError("Unexpected zip structure.")
    return tmp_root / top_dirs[0]


def list_images(folder: Path) -> List[Dict]:
    files: List[Dict] = []
    for p in sorted(folder.rglob("*")):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            files.append({"filename": p.name, "path": p.as_posix()})
    return files


def sanitize_name(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", (s or "").strip())


def row_name_from_filename(filename: str) -> str:
    stem = Path(filename).stem
    return stem.split("_", 1)[0] if "_" in stem else stem


def row_sort_key(row: str) -> Tuple[int, float, str]:
    """
    Stable ordering for grid/DF:
      neg -> numeric loads -> CC/K/dip -> pos -> everything else.
    """
    r = (row or "").strip()
    loads_order = {
        "neg": (0, -1.0),
        "1e5": (1, 1e5),
        "1.5e5": (2, 1.5e5),
        "5e5": (3, 5e5),
        "1e6": (4, 1e6),
        "5e6": (5, 5e6),
        "1e7": (6, 1e7),
        "cc": (7, 7e7),
        "k": (8, 8e7),
        "dip": (9, 9e7),
        "pos": (10, 1e12),
    }
    key = loads_order.get(r.lower())
    if key:
        return (0, key[0], r.lower())
    m = re.match(r"^(\d+(?:\.\d+)?)e(\d+)$", r.lower())
    if m:
        base = float(m.group(1))
        exp = float(m.group(2))
        return (1, base * (10**exp), r.lower())
    return (2, float("inf"), r.lower())


# ----------------- 1) load repo (names+paths) -----------------
base_dir = Path("/content") if Path("/content").exists() else Path.cwd()
zip_bytes = _download_repo_zip(OWNER, REPO, BRANCH)
REPO_DIR = _extract_zip_to_tmp(zip_bytes, base_dir)
DB_DIR = REPO_DIR / BASE_REL
assert DB_DIR.exists(), f"Missing path: {DB_DIR}"

DATASETS: Dict[str, List[Dict]] = {}
for sub in sorted([p for p in DB_DIR.iterdir() if p.is_dir()], key=lambda p: p.name.lower()):
    DATASETS[sub.name] = list_images(sub)

payload = {
    "repo_dir": str(REPO_DIR),
    "db_dir": str(DB_DIR),
    "folders": [{"name": k, "files": v} for k, v in DATASETS.items()],
}

# ----------------- 2) Flask app (new tab GUI) -----------------
from flask import Flask, jsonify, request, Response  # noqa

app = Flask(__name__)

_today = datetime.date.today()
DATE_FOLDER = f"{_today.month}-{_today.day}-{_today.year}"

TEMPLATE_HTML = r"""
<!doctype html>
<html><head><meta charset="utf-8" />
<meta name="viewport" content="width=device-width,initial-scale=1" />
<title>UW-LFA Dataset ROI</title>
<style>
 body{font:14px/1.45 system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,sans-serif;background:#0b0d12;color:#e6edf3;margin:0}
 .wrap{padding:16px 20px}
 .btn{background:#2a6de0;border:none;color:#fff;padding:10px 14px;border-radius:8px;cursor:pointer;font-weight:600}
 .btn2{background:#334155}
 label{opacity:.8;margin-right:8px}
 select, input[type=text], input[type=number]{background:#0f1623;border:1px solid #263145;color:#e6edf3;border-radius:8px;padding:7px 8px}
 .row{display:flex;gap:10px;align-items:center;flex-wrap:wrap;margin:10px 0}
 #imgWrap{position:relative;display:inline-block;border:1px solid #263145;border-radius:10px;overflow:hidden;background:#0f1623;max-width:720px;width:100%}
 #img{display:block;width:100%;height:auto}
 .rect{position:absolute;border:2px dashed #82aaff;background:rgba(130,170,255,.12);pointer-events:none}
 pre{background:#141824;border-radius:10px;padding:12px;overflow:auto}
 .ok{background:#2ea043}
 .panel{border:1px solid #263145;border-radius:10px;padding:12px;background:#0f1623}
 .mt{margin-top:18px}
 .muted{opacity:.8}
 .mono{font-family:ui-monospace,SFMono-Regular,Menlo,Monaco,Consolas,monospace}
 .loading{opacity:.6}
 #sbrCanvas{width:100%;max-width:960px;height:320px;border:1px solid #263145;border-radius:10px;background:#0b1020;display:none}
 .pill{display:inline-flex;gap:8px;align-items:center;padding:6px 10px;border:1px solid #263145;border-radius:999px;background:#0b1020}
</style>
</head>
<body>
<div class="wrap">
  <h2>UW-LFA Dataset ROI</h2>

  <div class="panel" style="display:flex;gap:10px;align-items:center;flex-wrap:wrap">
    <span class="muted">Saving to:</span>
    <input id="savePath" class="mono" type="text" style="min-width:360px;flex:1" readonly />
    <button id="copyPath" class="btn">Copy</button>
  </div>

  <div class="row" style="margin-top:10px">
    <button id="dl" class="btn">Download JSON (names + paths)</button>
  </div>

  <div class="panel">
    <div class="row">
      <label>Dataset (CSV name):</label>
      <select id="ds"></select>
      <button id="load" class="btn">Load dataset</button>

      <label style="margin-left:18px">Image:</label>
      <select id="imgsel"></select>

      <label style="margin-left:18px">Row:</label>
      <input id="rowName" class="mono" type="text" style="width:140px" readonly />
    </div>

    <div id="imgWrap">
      <img id="img" alt="(no image)" />
      <div id="rect" class="rect" style="display:none"></div>
    </div>

    <div class="row">
      <button id="select" class="btn">Select ROI</button>
      <button id="rotate" class="btn">⟳ Rotate 90°</button>
      <button id="reset"  class="btn">Reset View</button>

      <button id="save" class="btn">Save ROI (auto row)</button>
      <button id="export" class="btn ok">Export CSV for Dataset</button>
    </div>

    <div class="panel mt">
      <h3>ROI Grid (by Date + Dataset)</h3>
      <div class="row">
        <label>Date:</label>
        <select id="dfDate"></select>
        <button id="refreshDfDates" class="btn">↻</button>

        <label style="margin-left:12px">Dataset:</label>
        <select id="gridDataset"></select>

        <button id="plotGrid" class="btn">Plot Grid</button>
      </div>
      <img id="dfGrid" alt="ROI grid"
           style="display:none;max-width:960px;width:100%;border:1px solid #263145;border-radius:10px" />
    </div>

    <div class="panel mt">
      <h3>SBR Profile (Blue + Red → green peak → yellow baseline → save into CSV)</h3>
      <div class="row">
        <label>Row index</label>
        <input id="sbrIndex" type="number" min="0" value="0" style="width:80px" />

        <button id="plotSBR" class="btn">Load SBR</button>
        <button id="resetLines" class="btn btn2">Reset Lines</button>

        <span class="pill"><span class="muted">Blue</span><span id="blueInfo" class="mono">-</span></span>
        <span class="pill"><span class="muted">Red</span><span id="redInfo" class="mono">-</span></span>
        <span class="pill"><span class="muted">Peak</span><span id="greenInfo" class="mono">-</span></span>
        <span class="pill"><span class="muted">Baseline</span><span id="yellowInfo" class="mono">-</span></span>
      </div>

      <canvas id="sbrCanvas"></canvas>
      <div id="sbrMsg" class="muted" style="margin-top:8px"></div>
    </div>
  </div>

  <pre id="log"></pre>
</div>

<script>
const LOG = (m)=>{const el=document.getElementById('log'); el.textContent += m + "\\n"; el.scrollTop = el.scrollHeight;};
const MSG = (m)=>{document.getElementById('sbrMsg').textContent = m || "";};

async function fillSavePath(){
  try{
    const r = await fetch('/export_info'); const j = await r.json();
    document.getElementById('savePath').value = j.full;
  }catch(e){}
}
document.getElementById('copyPath').onclick = ()=>{
  const el = document.getElementById('savePath'); el.select(); document.execCommand('copy');
};

const DS = document.getElementById('ds');
const IMGSEL = document.getElementById('imgsel');
const IMG = document.getElementById('img');
const RECT = document.getElementById('rect');
const ROWNAME = document.getElementById('rowName');
const GRID_DS = document.getElementById('gridDataset');

let mouseDown=false, startX=0, startY=0;
let currentDataset = null, currentIndex = null;
let imgBusy = false;
let roiMode=false;

IMG.style.userSelect = 'none';
IMG.style.touchAction = 'none';

function hardResetROI(){
  RECT.style.display='none';
  RECT.style.left = RECT.style.top = '0px';
  RECT.style.width = RECT.style.height = '0px';
  mouseDown = false; roiMode = false;
  IMG.style.cursor = 'default';
}

function setImgSrc() {
  if (!currentDataset || currentIndex===null) return;
  if (imgBusy) return;
  imgBusy = true;
  IMG.classList.add('loading');
  const url = `/image?dataset=${encodeURIComponent(currentDataset)}&index=${currentIndex}&t=${Date.now()}`;
  const done = ()=>{ imgBusy=false; IMG.classList.remove('loading'); };
  IMG.onload = done; IMG.onerror = done;
  IMG.src = url;
}

async function fetchDatasets(){
  const r = await fetch('/datasets'); const j = await r.json();
  const opts = j.names.map(n=>`<option>${n}</option>`).join('');
  DS.innerHTML = opts;
  GRID_DS.innerHTML = opts;
  LOG("Datasets: " + j.names.join(", "));
}

async function loadDataset(){
  currentDataset = DS.value; currentIndex = null;
  const r = await fetch('/images?dataset=' + encodeURIComponent(currentDataset));
  const j = await r.json();
  if (!j.files.length){ IMGSEL.innerHTML=""; IMG.removeAttribute('src'); ROWNAME.value=""; LOG("No images."); return; }

  IMGSEL.innerHTML = j.files.map((f,i)=>`<option value="${i}">${f.filename} (${f.width}×${f.height})</option>`).join('');
  hardResetROI();
  currentIndex = 0;
  ROWNAME.value = j.files[0].row;
  setImgSrc();
  LOG(`Loaded dataset: ${currentDataset} (${j.files.length} files)`);
}

IMGSEL.onchange = async e => {
  currentIndex = parseInt(e.target.value,10);
  hardResetROI();
  const r = await fetch(`/row_name?dataset=${encodeURIComponent(currentDataset)}&index=${currentIndex}`);
  const j = await r.json();
  ROWNAME.value = j.row || '';
  setImgSrc();
};

document.getElementById('load').onclick = loadDataset;

document.getElementById('dl').onclick = () => {
  const blob = new Blob([JSON.stringify(__PAYLOAD__, null, 2)], {type:'application/json'});
  const url = URL.createObjectURL(blob); const a = document.createElement('a');
  a.href = url; a.download = 'uw_lfa_folders.json'; a.click(); setTimeout(()=>URL.revokeObjectURL(url), 500);
};

document.getElementById('select').onclick = () => {
  roiMode = !roiMode;
  RECT.style.display = roiMode ? 'block' : 'none';
  IMG.style.cursor = roiMode ? 'crosshair' : 'default';
  LOG("ROI: " + (roiMode ? "ON" : "OFF"));
};

document.getElementById('reset').onclick  = async () => {
  hardResetROI();
  try{
    const r = await fetch('/reset_view', {method:'POST'}); const j = await r.json(); LOG(j.msg || 'Reset.');
  }catch(e){ LOG('Reset failed: ' + e); }
  setImgSrc();
};

document.getElementById('rotate').onclick = async () => {
  const r = await fetch('/rotate', {method:'POST'}); const j = await r.json(); LOG(j.msg); setImgSrc();
};

function clamp(v,min,max){ return Math.max(min, Math.min(max, v)); }
function posInImg(e){
  const r = IMG.getBoundingClientRect();
  const x = clamp(e.clientX - r.left, 0, r.width);
  const y = clamp(e.clientY - r.top , 0, r.height);
  return {x, y, rect: r};
}

IMG.addEventListener('dragstart', e => e.preventDefault());
IMG.addEventListener('mousedown', e=>{
  if(!roiMode) return; e.preventDefault();
  const p = posInImg(e);
  mouseDown = true; startX = p.x; startY = p.y;
  RECT.style.left = startX + 'px'; RECT.style.top  = startY + 'px';
  RECT.style.width = '0px'; RECT.style.height = '0px';
});

window.addEventListener('mousemove', e=>{
  if(!roiMode || !mouseDown) return;
  const p = posInImg(e);
  const left = Math.min(startX, p.x), top = Math.min(startY, p.y);
  const w = Math.abs(p.x - startX), h = Math.abs(p.y - startY);
  RECT.style.left=left+'px'; RECT.style.top=top+'px'; RECT.style.width=w+'px'; RECT.style.height=h+'px';
});

window.addEventListener('mouseup', async e=>{
  if(!roiMode || !mouseDown) return; mouseDown = false;
  const r = IMG.getBoundingClientRect();
  const curr = posInImg(e);
  const left = Math.min(startX, curr.x), top  = Math.min(startY, curr.y);
  const w = Math.abs(curr.x - startX), h = Math.abs(curr.y - startY);
  if (w < 3 || h < 3) { LOG("Selection too small."); return; }
  const nx = left / r.width, ny = top  / r.height, nw = w / r.width, nh = h / r.height;
  const tw = Math.round(r.width), th = Math.round(r.height);
  const resp = await fetch('/apply_crop_zoom', {
    method:'POST', headers:{'Content-Type':'application/json'},
    body: JSON.stringify({nx, ny, nw, nh, tw, th})
  });
  const j = await resp.json(); LOG(j.msg); setImgSrc();
});

document.getElementById('save').onclick = async () => {
  if (!currentDataset || currentIndex===null) return;
  const r = await fetch('/save_roi', {
    method:'POST', headers:{'Content-Type':'application/json'},
    body: JSON.stringify({ dataset: currentDataset, index: currentIndex })
  });
  const j = await r.json(); LOG(j.msg);
};

document.getElementById('export').onclick = async () => {
  if (!currentDataset) return;
  const r = await fetch('/export_df', {
    method:'POST', headers:{'Content-Type':'application/json'},
    body: JSON.stringify({ dataset: currentDataset })
  });
  const j = await r.json(); LOG(j.msg);
};

// dates + grid
const DFGRID  = document.getElementById('dfGrid');
const DFDATES = document.getElementById('dfDate');

async function fetchDfDates(){
  const r = await fetch('/exports'); const j = await r.json();
  if (!j.dates.length){ DFDATES.innerHTML = '<option>(none)</option>'; DFGRID.style.display='none'; return; }
  DFDATES.innerHTML = j.dates.map(d=>`<option>${d}</option>`).join('');
}
document.getElementById('refreshDfDates').onclick = fetchDfDates;

document.getElementById('plotGrid').onclick = () => {
  const date = DFDATES.value;
  const ds = GRID_DS.value;
  if(!date || date === '(none)'){ LOG('No export dates found.'); return; }
  if(!ds){ LOG('Pick a dataset.'); return; }
  DFGRID.src = `/analyze_grid?date=${encodeURIComponent(date)}&dataset=${encodeURIComponent(ds)}&t=${Date.now()}`;
  DFGRID.style.display = 'block';
};

// ----------------- interactive SBR -----------------
const canvas = document.getElementById('sbrCanvas');
const ctx = canvas.getContext('2d');
const blueInfo = document.getElementById('blueInfo');
const redInfo = document.getElementById('redInfo');
const greenInfo = document.getElementById('greenInfo');
const yellowInfo = document.getElementById('yellowInfo');

let sbrProfile = null;  // float[]
let sbrN = 0;
let xBlue = null;
let xRed = null;
let xPeak = null;
let peakVal = null;
let baselineVal = null;
let baselineXHalf = null;
let sbrRowName = null;

let plotGeom = null;

function setCanvasSize(){
  const rect = canvas.getBoundingClientRect();
  const dpr = window.devicePixelRatio || 1;
  canvas.width = Math.max(1, Math.floor(rect.width * dpr));
  canvas.height = Math.max(1, Math.floor(rect.height * dpr));
  ctx.setTransform(dpr,0,0,dpr,0,0);
}

function resetLines(){
  xBlue = null; xRed = null; xPeak = null;
  peakVal = null;
  baselineVal = null;
  baselineXHalf = null;
  blueInfo.textContent = "-";
  redInfo.textContent = "-";
  greenInfo.textContent = "-";
  yellowInfo.textContent = "-";
  MSG("Click two points on the graph (x must be >=5 and <= n-6).");
  drawSBR();
}
document.getElementById('resetLines').onclick = resetLines;

function niceNum(range, round) {
  const exponent = Math.floor(Math.log10(range || 1));
  const fraction = range / Math.pow(10, exponent);
  let niceFraction;
  if (round) {
    if (fraction < 1.5) niceFraction = 1;
    else if (fraction < 3) niceFraction = 2;
    else if (fraction < 7) niceFraction = 5;
    else niceFraction = 10;
  } else {
    if (fraction <= 1) niceFraction = 1;
    else if (fraction <= 2) niceFraction = 2;
    else if (fraction <= 5) niceFraction = 5;
    else niceFraction = 10;
  }
  return niceFraction * Math.pow(10, exponent);
}

function computeTicks(min, max, maxTicks = 5) {
  const range = niceNum(max - min, false);
  const step = niceNum(range / Math.max(1, (maxTicks - 1)), true);
  const graphMin = Math.floor(min / step) * step;
  const graphMax = Math.ceil(max / step) * step;

  const ticks = [];
  for (let v = graphMin; v <= graphMax + 0.5 * step; v += step) ticks.push(v);
  return { ticks, graphMin, graphMax, step };
}

function formatTick(v, step){
  const absStep = Math.abs(step || 0);
  let decimals = 0;
  if (absStep > 0) {
    decimals = Math.max(0, -Math.floor(Math.log10(absStep)) + 1);
    decimals = Math.min(decimals, 6);
  }
  let s = v.toFixed(decimals);
  s = s.replace(/\.?0+$/, '');
  return s;
}

function drawSBR(){
  if (!sbrProfile || !sbrProfile.length){
    canvas.style.display = 'none';
    plotGeom = null;
    return;
  }
  canvas.style.display = 'block';
  setCanvasSize();

  const rect = canvas.getBoundingClientRect();
  const W = rect.width;
  const H = rect.height;
  ctx.clearRect(0,0,W,H);

  const padL = 62, padR = 18, padT = 34, padB = 46;

  let yMin = Infinity, yMax = -Infinity;
  for (const v of sbrProfile){ if (v<yMin) yMin=v; if (v>yMax) yMax=v; }
  if (!isFinite(yMin) || !isFinite(yMax)){ yMin=0; yMax=1; }

  // include baseline in bounds if present
  if (baselineVal !== null && isFinite(baselineVal)){
    yMin = Math.min(yMin, baselineVal);
    yMax = Math.max(yMax, baselineVal);
  }

  const yPad = (yMax - yMin) * 0.08 || 0.01;
  yMin -= yPad; yMax += yPad;

  const maxX = sbrN - 1;
  const xTicks = [];
  const xStep = 50;
  for (let x=0; x<=maxX; x+=xStep) xTicks.push(x);
  if (xTicks[xTicks.length-1] !== maxX) xTicks.push(maxX);

  const yt = computeTicks(yMin, yMax, 5);

  const xToPx = (x) => padL + (x / (sbrN - 1)) * (W - padL - padR);
  const yToPx = (y) => (H - padB) - ((y - yt.graphMin) / (yt.graphMax - yt.graphMin + 1e-9)) * (H - padT - padB);

  plotGeom = { padL, padR, padT, padB, W, H, xToPx, yToPx };

  // gridlines + tick labels
  ctx.lineWidth = 1;
  ctx.strokeStyle = "rgba(230,237,243,0.18)";
  ctx.font = "12px ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, monospace";
  ctx.fillStyle = "#e6edf3";

  ctx.textAlign = "center";
  ctx.textBaseline = "top";
  for (const xtv of xTicks){
    const x = xToPx(xtv);
    ctx.beginPath();
    ctx.moveTo(x, padT);
    ctx.lineTo(x, H-padB);
    ctx.stroke();
    ctx.fillText(String(xtv), x, H-padB+8);
  }

  ctx.textAlign = "right";
  ctx.textBaseline = "middle";
  for (const ytv of yt.ticks){
    const y = yToPx(ytv);
    ctx.beginPath();
    ctx.moveTo(padL, y);
    ctx.lineTo(W-padR, y);
    ctx.stroke();
    ctx.fillText(formatTick(ytv, yt.step), padL-10, y);
  }

  // axes box
  ctx.strokeStyle = "rgba(230,237,243,0.35)";
  ctx.lineWidth = 1.25;
  ctx.beginPath();
  ctx.rect(padL, padT, (W-padL-padR), (H-padT-padB));
  ctx.stroke();

  // title
  ctx.fillStyle = "#e6edf3";
  ctx.font = "600 16px system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,sans-serif";
  const title = `SBR – ${(window.DF_DATASET || "")} – row ${sbrRowName || ""}`;
  ctx.textAlign = "center";
  ctx.textBaseline = "top";
  ctx.fillText(title, W/2, 8);

  // axis labels
  ctx.fillStyle = "#e6edf3";
  ctx.font = "13px system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,sans-serif";

  ctx.textAlign = "center";
  ctx.textBaseline = "top";
  ctx.fillText("Position along strip", (padL + (W-padR))/2, H-28);

  ctx.save();
  ctx.translate(18, (padT + (H-padB))/2);
  ctx.rotate(-Math.PI/2);
  ctx.textAlign = "center";
  ctx.textBaseline = "top";
  ctx.fillText("Intensity (smoothed)", 0, 0);
  ctx.restore();

  // shaded ±5 windows (behind line)
  function shadeWindow(ix, rgba){
    if (ix === null) return;
    const left = xToPx(ix - 5);
    const right = xToPx(ix + 5);
    ctx.fillStyle = rgba;
    ctx.fillRect(left, padT, (right-left), (H-padB-padT));
  }
  shadeWindow(xBlue, "rgba(59,130,246,0.12)");
  shadeWindow(xRed,  "rgba(239,68,68,0.12)");

  // baseline (yellow horizontal line) behind the profile line but above shading
  if (baselineVal !== null && isFinite(baselineVal)){
    const yb = yToPx(baselineVal);
    ctx.strokeStyle = "#eab308"; // yellow
    ctx.lineWidth = 2;
    ctx.beginPath();
    ctx.moveTo(padL, yb);
    ctx.lineTo(W-padR, yb);
    ctx.stroke();
  }

  // profile line
  ctx.strokeStyle = "#9bbcff";
  ctx.lineWidth = 2;
  ctx.beginPath();
  for (let i=0;i<sbrN;i++){
    const x = xToPx(i);
    const y = yToPx(sbrProfile[i]);
    if (i===0) ctx.moveTo(x,y); else ctx.lineTo(x,y);
  }
  ctx.stroke();

  // vertical lines
  function drawV(ix, color){
    const x = xToPx(ix);
    ctx.strokeStyle = color;
    ctx.lineWidth = 2;
    ctx.beginPath();
    ctx.moveTo(x, padT);
    ctx.lineTo(x, H-padB);
    ctx.stroke();
  }
  if (xBlue !== null) drawV(xBlue, "#3b82f6");
  if (xRed !== null)  drawV(xRed,  "#ef4444");
  if (xPeak !== null) drawV(xPeak, "#22c55e"); // green peak

  // legend
  ctx.font = "12px system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,sans-serif";
  ctx.textAlign = "left";
  ctx.textBaseline = "top";
  let lx = padL + 8, ly = padT + 8;
  if (xBlue !== null){
    ctx.fillStyle = "#3b82f6"; ctx.fillRect(lx, ly+4, 10, 2);
    ctx.fillStyle = "#e6edf3"; ctx.fillText(`Blue @ ${xBlue} (±5)`, lx+14, ly);
    ly += 18;
  }
  if (xRed !== null){
    ctx.fillStyle = "#ef4444"; ctx.fillRect(lx, ly+4, 10, 2);
    ctx.fillStyle = "#e6edf3"; ctx.fillText(`Red @ ${xRed} (±5)`, lx+14, ly);
    ly += 18;
  }
  if (xPeak !== null){
    ctx.fillStyle = "#22c55e"; ctx.fillRect(lx, ly+4, 10, 2);
    ctx.fillStyle = "#e6edf3"; ctx.fillText(`Peak @ ${xPeak}`, lx+14, ly);
    ly += 18;
  }
  if (baselineVal !== null && isFinite(baselineVal)){
    ctx.fillStyle = "#eab308"; ctx.fillRect(lx, ly+4, 10, 2);
    const halfTxt = (baselineXHalf !== null) ? ` (x_half=${baselineXHalf})` : "";
    ctx.fillStyle = "#e6edf3"; ctx.fillText(`Baseline = ${Number(baselineVal).toFixed(6)}${halfTxt}`, lx+14, ly);
  }
}

function pxToIndex(clientX){
  if (!plotGeom) return 0;
  const rect = canvas.getBoundingClientRect();
  const { padL, padR, W } = plotGeom;
  const x = Math.max(padL, Math.min(W-padR, clientX - rect.left));
  const t = (x - padL) / (W - padL - padR);
  return Math.round(t * (sbrN - 1));
}

function validateIndex(ix){
  if (ix < 5 || ix > (sbrN - 6)){
    return {ok:false, msg:`Pick a different point: need 5 points on each side (valid: 5..${sbrN-6}). You picked ${ix}.`};
  }
  return {ok:true, msg:""};
}

canvas.addEventListener('click', async (e)=>{
  if (!sbrProfile) return;
  const ix = pxToIndex(e.clientX);
  const v = validateIndex(ix);
  if (!v.ok){ MSG(v.msg); return; }

  if (xBlue === null){
    xBlue = ix;
    blueInfo.textContent = `${ix} (window ${ix-5}-${ix+5})`;
    MSG("Blue set. Click second point for Red.");
    drawSBR();
    return;
  }
  if (xRed === null){
    xRed = ix;
    redInfo.textContent = `${ix} (window ${ix-5}-${ix+5})`;
    drawSBR();
    MSG("Red set. Computing medians + peak + baseline, updating CSV...");
    const idx = document.getElementById('sbrIndex').value || '0';
    const r = await fetch('/update_medians', {
      method:'POST', headers:{'Content-Type':'application/json'},
      body: JSON.stringify({ index: parseInt(idx,10), x_blue: xBlue, x_red: xRed })
    });
    const j = await r.json();
    if (j.ok){
      xPeak = j.peak_x;
      peakVal = j.peak_value;
      baselineVal = j.baseline_value;
      baselineXHalf = j.baseline_xhalf;

      greenInfo.textContent = `${xPeak} (val ${Number(peakVal).toFixed(6)})`;
      yellowInfo.textContent = `${Number(baselineVal).toFixed(6)} (x_half ${baselineXHalf})`;
      drawSBR();
    }
    MSG(j.msg || (j.ok ? "Updated CSV." : "Failed."));
    return;
  }
  MSG("You already selected 2 points. Click Reset Lines to choose different ones.");
});

document.getElementById('plotSBR').onclick = async () => {
  const idx = document.getElementById('sbrIndex').value || '0';
  const r = await fetch(`/sbr_profile_json?index=${encodeURIComponent(idx)}&t=${Date.now()}`);
  const j = await r.json();
  if (!j.ok){
    MSG(j.msg || "Failed to load SBR. Did you click Export CSV first?");
    canvas.style.display = 'none';
    plotGeom = null;
    return;
  }
  window.DF_DATASET = j.dataset || "";

  sbrProfile = j.profile;
  sbrN = j.n;
  sbrRowName = j.row_name;
  resetLines();
};

(async function init(){ await fillSavePath(); await fetchDatasets(); await fetchDfDates(); })();
</script>
</body></html>
"""

# ----------------- server state -----------------
CURRENT_IMAGE = None
ORIGINAL_IMAGE = None

TARGET_H, TARGET_W = 70, 270

ROI_BY_DATASET_ROW: Dict[Tuple[str, str], np.ndarray] = {}
ROI_SAVE_COUNT: Dict[Tuple[str, str], int] = {}

DF_GRAY: Optional["pd.DataFrame"] = None
DF_DATASET: Optional[str] = None
DF_CSV_PATH: Optional[Path] = None


def _load_image_by(dataset: str, index: int):
    files = DATASETS.get(dataset, [])
    if index < 0 or index >= len(files):
        return None
    return cv2.imread(files[index]["path"], cv2.IMREAD_UNCHANGED)


def _normalized_roi_from_current() -> np.ndarray:
    if CURRENT_IMAGE is None:
        raise ValueError("No current image.")
    gray = CURRENT_IMAGE if CURRENT_IMAGE.ndim == 2 else cv2.cvtColor(CURRENT_IMAGE, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)
    return (255.0 - gray.astype(np.float32)) / 255.0


def _dataset_rows_from_files(ds_raw: str) -> List[str]:
    rows = [sanitize_name(row_name_from_filename(f["filename"])) for f in DATASETS.get(ds_raw, [])]
    return sorted(set(rows), key=row_sort_key)


def _png_message(msg: str) -> Response:
    fig, ax = plt.subplots(figsize=(7, 2.2))
    ax.text(0.02, 0.6, msg, ha="left", va="center", fontsize=10)
    ax.set_axis_off()
    buf = io.BytesIO()
    fig.tight_layout()
    fig.savefig(buf, format="png", dpi=150)
    plt.close(fig)
    buf.seek(0)
    return Response(buf.getvalue(), mimetype="image/png")


def _first_match(base_dir: Path, dataset: str, row: str) -> Optional[str]:
    folder = base_dir / dataset
    if not folder.exists():
        return None
    prefix = f"ROI_{row}_"
    for f in sorted(folder.glob(prefix + "*.tif*")):
        return str(f)
    return None


def _parse_pixels_to_roi(px: str) -> Optional[np.ndarray]:
    if not px:
        return None
    vals = np.fromstring(px, sep=",", dtype=np.float32)
    if vals.size != TARGET_H * TARGET_W:
        return None
    return vals.reshape((TARGET_H, TARGET_W))


def _sbr_profile_from_roi(roi: np.ndarray) -> np.ndarray:
    prof = np.mean(roi, axis=0).astype(np.float32)
    if prof.size >= 7:
        return savgol_filter(prof, 5, 2).astype(np.float32)
    return prof


def _median_window(profile: np.ndarray, x: int, half_window: int = 5) -> float:
    lo = x - half_window
    hi = x + half_window
    if lo < 0 or hi >= profile.size:
        raise ValueError(f"Index {x} out of range for ±{half_window} window (n={profile.size}).")
    return float(np.median(profile[lo : hi + 1]))


def _peak_between(profile: np.ndarray, x1: int, x2: int) -> Tuple[int, float]:
    lo = int(min(x1, x2))
    hi = int(max(x1, x2))
    seg = profile[lo : hi + 1]
    if seg.size == 0:
        raise ValueError("Empty peak segment.")
    j = int(np.argmax(seg))
    peak_x = lo + j
    peak_val = float(seg[j])
    return int(peak_x), float(peak_val)


def _baseline_value(
    x_b: int,  # blue x
    x_a: int,  # red x
    b: float,  # blue median
    a: float,  # red median
) -> Tuple[float, float]:
    """
    Baseline equation given:
      b + [(a-b)*(x_half-x_b)/(x_a-xb)]
    where x_half = (x_b + x_a)/2

    Returns (x_half, baseline_value).
    """
    if x_a == x_b:
        raise ValueError("x_a equals x_b; cannot compute baseline.")
    x_half = (float(x_b) + float(x_a)) / 2.0
    baseline = float(b) + ((float(a) - float(b)) * (x_half - float(x_b)) / (float(x_a) - float(x_b)))
    return x_half, baseline


# ----------------- endpoints -----------------
@app.route("/")
def home():
    html = TEMPLATE_HTML.replace("__PAYLOAD__", json.dumps(payload))
    return Response(html, mimetype="text/html")


@app.route("/export_info")
def export_info():
    full = (EXPORT_ROOT / DATE_FOLDER).as_posix()
    return jsonify({"root": str(EXPORT_ROOT), "date": DATE_FOLDER, "full": full})


@app.route("/datasets")
def datasets():
    return jsonify({"names": list(DATASETS.keys())})


@app.route("/row_name")
def row_name():
    ds = request.args.get("dataset", "")
    idx = int(request.args.get("index", "0"))
    files = DATASETS.get(ds, [])
    if 0 <= idx < len(files):
        return jsonify({"row": row_name_from_filename(files[idx]["filename"])})
    return jsonify({"row": ""})


@app.route("/images")
def images():
    ds = request.args.get("dataset", "")
    files = DATASETS.get(ds, [])
    files_out = []
    for f in files:
        w = h = c = 0
        try:
            img = cv2.imread(f["path"], cv2.IMREAD_UNCHANGED)
            if img is not None:
                h, w = img.shape[:2]
                c = 1 if img.ndim == 2 else img.shape[2]
        except Exception:
            pass
        files_out.append(
            {
                **f,
                "width": int(w),
                "height": int(h),
                "channels": int(c),
                "row": row_name_from_filename(f["filename"]),
            }
        )
    return jsonify({"files": files_out})


@app.route("/image")
def image():
    global CURRENT_IMAGE, ORIGINAL_IMAGE
    ds = request.args.get("dataset", "")
    idx = int(request.args.get("index", "0"))

    key = (ds, idx)
    if getattr(app, "_current_key", None) != key:
        img = _load_image_by(ds, idx)
        if img is None:
            img = np.zeros((40, 120, 3), np.uint8)
        ORIGINAL_IMAGE = img.copy()
        CURRENT_IMAGE = img.copy()
        app._current_key = key

    bgr = CURRENT_IMAGE
    rgb = bgr if bgr.ndim == 2 else cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    _, buf = cv2.imencode(".png", rgb)
    return Response(io.BytesIO(buf).getvalue(), mimetype="image/png")


@app.route("/reset_view", methods=["POST"])
def reset_view():
    global CURRENT_IMAGE, ORIGINAL_IMAGE
    if ORIGINAL_IMAGE is not None:
        CURRENT_IMAGE = ORIGINAL_IMAGE.copy()
        return jsonify({"ok": True, "msg": "View reset."})
    return jsonify({"ok": False, "msg": "No image to reset."})


@app.route("/rotate", methods=["POST"])
def rotate():
    global CURRENT_IMAGE, ORIGINAL_IMAGE
    if CURRENT_IMAGE is None or ORIGINAL_IMAGE is None:
        return jsonify({"ok": False, "msg": "No image loaded."})
    CURRENT_IMAGE = cv2.rotate(CURRENT_IMAGE, cv2.ROTATE_90_COUNTERCLOCKWISE)
    ORIGINAL_IMAGE = cv2.rotate(ORIGINAL_IMAGE, cv2.ROTATE_90_COUNTERCLOCKWISE)
    return jsonify({"ok": True, "msg": "Rotated 90° CCW (persistent)."})


@app.route("/apply_crop_zoom", methods=["POST"])
def apply_crop_zoom():
    global CURRENT_IMAGE, ORIGINAL_IMAGE
    if CURRENT_IMAGE is None or ORIGINAL_IMAGE is None:
        return jsonify({"ok": False, "msg": "No image loaded."})

    d = request.get_json(force=True)
    nx = float(d.get("nx", 0.0))
    ny = float(d.get("ny", 0.0))
    nw = float(d.get("nw", 0.0))
    nh = float(d.get("nh", 0.0))
    tW = int(d.get("tw", ORIGINAL_IMAGE.shape[1]))
    tH = int(d.get("th", ORIGINAL_IMAGE.shape[0]))

    H, W = CURRENT_IMAGE.shape[:2]
    x1 = int(max(0, min(W - 1, round(nx * W))))
    y1 = int(max(0, min(H - 1, round(ny * H))))
    x2 = int(max(0, min(W, round((nx + nw) * W))))
    y2 = int(max(0, min(H, round((ny + nh) * H))))
    if x2 <= x1 or y2 <= y1:
        return jsonify({"ok": False, "msg": "Empty selection."})

    crop = CURRENT_IMAGE[y1:y2, x1:x2]
    upscale = (crop.shape[1] < tW) or (crop.shape[0] < tH)
    interp = cv2.INTER_CUBIC if upscale else cv2.INTER_AREA
    CURRENT_IMAGE = cv2.resize(crop, (tW, tH), interpolation=interp)
    return jsonify({"ok": True, "msg": f"Cropped [{x1}:{x2}]×[{y1}:{y2}] → {tW}×{tH}."})


@app.route("/save_roi", methods=["POST"])
def save_roi():
    data = request.get_json(force=True)
    ds_raw = data.get("dataset", "")
    idx = int(data.get("index", 0))

    if ds_raw not in DATASETS:
        return jsonify({"ok": False, "msg": f"Unknown dataset: {ds_raw}"})
    files = DATASETS.get(ds_raw, [])
    if not (0 <= idx < len(files)):
        return jsonify({"ok": False, "msg": "Bad image index."})

    ds = sanitize_name(ds_raw)
    row = sanitize_name(row_name_from_filename(files[idx]["filename"]))

    try:
        roi = _normalized_roi_from_current()
    except Exception as e:
        return jsonify({"ok": False, "msg": f"ROI not available: {e}"})

    ROI_BY_DATASET_ROW[(ds, row)] = roi

    out_date_dir = EXPORT_ROOT / DATE_FOLDER
    out_ds_dir = out_date_dir / ds
    out_ds_dir.mkdir(parents=True, exist_ok=True)

    key = (ds, row)
    ROI_SAVE_COUNT[key] = ROI_SAVE_COUNT.get(key, 0) + 1
    n = ROI_SAVE_COUNT[key]
    out_path = out_ds_dir / f"ROI_{row}_{n}.tiff"
    cv2.imwrite(str(out_path), (roi * 255.0).astype(np.uint8))

    return jsonify({"ok": True, "msg": f"Saved ROI → {out_path} (dataset={ds_raw}, row={row})"})


@app.route("/export_df", methods=["POST"])
def export_df():
    """
    Writes CSV: .../<date>/<dataset>.csv
    Columns:
      pixels, shape,
      x_blue, median_blue,
      x_red,  median_red,
      peak_x, peak_value,
      baseline_xhalf, baseline_value
    """
    global DF_GRAY, DF_DATASET, DF_CSV_PATH

    data = request.get_json(force=True) if request.data else {}
    ds_raw = (data.get("dataset") or "").strip()
    if ds_raw not in DATASETS:
        return jsonify({"ok": False, "msg": f"Unknown dataset: {ds_raw}", "html": ""})

    ds = sanitize_name(ds_raw)
    rows = _dataset_rows_from_files(ds_raw)

    def arr_to_str(a: Optional[np.ndarray]) -> str:
        if isinstance(a, np.ndarray) and a.size > 0:
            return ",".join(map(str, a.flatten()))
        return ""

    def arr_shape(a: Optional[np.ndarray]) -> str:
        if isinstance(a, np.ndarray) and a.size > 0:
            return f"{a.shape[0]}x{a.shape[1]}"
        return "0x0"

    records = []
    for row in rows:
        arr = ROI_BY_DATASET_ROW.get((ds, row))
        records.append(
            {
                "row": row,
                "pixels": arr_to_str(arr),
                "shape": arr_shape(arr),
                "x_blue": "",
                "median_blue": "",
                "x_red": "",
                "median_red": "",
                "peak_x": "",
                "peak_value": "",
                "baseline_xhalf": "",
                "baseline_value": "",
            }
        )

    df = pd.DataFrame.from_records(records).set_index("row")

    out_date_dir = EXPORT_ROOT / DATE_FOLDER
    out_date_dir.mkdir(parents=True, exist_ok=True)
    out_csv = out_date_dir / f"{ds}.csv"
    df.to_csv(out_csv)

    DF_GRAY = df.copy()
    DF_DATASET = ds
    DF_CSV_PATH = out_csv

    return jsonify({"ok": True, "msg": f"Wrote {out_csv} (rows={len(df)})", "html": ""})


@app.route("/exports")
def list_exports():
    dates = []
    if EXPORT_ROOT.exists():
        for p in sorted(EXPORT_ROOT.iterdir()):
            if p.is_dir():
                dates.append(p.name)
    return jsonify({"dates": dates})


@app.route("/analyze_grid")
def analyze_grid():
    try:
        date = (request.args.get("date", "") or "").strip()
        ds_raw = (request.args.get("dataset", "") or "").strip()

        if not date:
            return _png_message("Grid error: missing date.")
        if ds_raw not in DATASETS:
            return _png_message(f"Grid error: dataset not found:\n{ds_raw}")

        ds = sanitize_name(ds_raw)
        base = EXPORT_ROOT / date
        if not base.exists():
            return _png_message(f"Grid error: date folder not found:\n{base.as_posix()}")

        ds_dir = base / ds
        if not ds_dir.exists():
            return _png_message(
                "Grid error: dataset folder not found under date.\n"
                f"Expected:\n{ds_dir.as_posix()}\n\n"
                "Did you click 'Save ROI' for this dataset/date?"
            )

        rows = _dataset_rows_from_files(ds_raw)
        if not rows:
            return _png_message(f"Grid error: no rows inferred from dataset files:\n{ds_raw}")

        imgs: List[Optional[np.ndarray]] = []
        found_any = False
        for row in rows:
            path = _first_match(base, ds, row)
            if path:
                img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
                if img is not None:
                    imgs.append(img.astype(np.float32) / 255.0)
                    found_any = True
                    continue
            imgs.append(None)

        if not found_any:
            example_row = rows[0]
            return _png_message(
                "Grid error: no ROI TIFFs matched the expected pattern.\n"
                f"Looked under:\n{ds_dir.as_posix()}\n"
                f"Example pattern:\nROI_{example_row}_*.tiff"
            )

        nrows = len(rows)
        fig_w = 5.8
        fig_h = max(3.0, 0.65 * nrows)
        fig, axs = plt.subplots(nrows, 1, figsize=(fig_w, fig_h))
        axs = np.atleast_1d(axs).ravel()

        for i, row in enumerate(rows):
            ax = axs[i]
            if imgs[i] is not None:
                ax.imshow(imgs[i], cmap="gray", aspect="auto")
            ax.set_axis_off()
            ax.text(-0.03, 0.5, row, va="center", ha="right", transform=ax.transAxes, fontsize=9)

        plt.suptitle(f"{ds_raw} – ROIs from {date}", y=0.995, fontsize=12)
        plt.tight_layout()

        buf = io.BytesIO()
        plt.savefig(buf, format="png", dpi=150, bbox_inches="tight")
        plt.close(fig)
        buf.seek(0)
        return Response(buf.getvalue(), mimetype="image/png")

    except Exception as e:
        return _png_message(f"Grid crashed:\n{type(e).__name__}: {e}")


# ----------------- interactive SBR endpoints -----------------
@app.route("/sbr_profile_json")
def sbr_profile_json():
    try:
        idx = int(request.args.get("index", "0"))
    except ValueError:
        idx = 0

    if DF_GRAY is None or not isinstance(DF_GRAY, pd.DataFrame) or DF_GRAY.empty:
        return jsonify({"ok": False, "msg": "No exported CSV/DF yet. Click 'Export CSV for Dataset' first."})
    if idx < 0 or idx >= len(DF_GRAY):
        return jsonify({"ok": False, "msg": f"Row index out of range (0..{len(DF_GRAY)-1})."})

    row_name = str(DF_GRAY.index[idx])
    px = str(DF_GRAY.iloc[idx].get("pixels", "") or "")
    roi = _parse_pixels_to_roi(px)
    if roi is None:
        return jsonify({"ok": False, "msg": f"No ROI saved for row '{row_name}'. Save ROI first."})

    profile = _sbr_profile_from_roi(roi)
    return jsonify(
        {
            "ok": True,
            "dataset": DF_DATASET or "",
            "row_name": row_name,
            "n": int(profile.size),
            "profile": profile.astype(float).tolist(),
        }
    )


@app.route("/update_medians", methods=["POST"])
def update_medians():
    """
    POST {index, x_blue, x_red}
    Updates DF + rewrites CSV with:
      x_blue, median_blue, x_red, median_red,
      peak_x, peak_value,
      baseline_xhalf, baseline_value

    Baseline uses your equation:
      baseline = b + [(a-b)*(x_half-x_b)/(x_a-x_b)]
      x_b = blue x
      x_a = red x
      x_half = (x_b + x_a)/2
      b = median_blue
      a = median_red
    """
    global DF_GRAY, DF_CSV_PATH

    if DF_GRAY is None or DF_CSV_PATH is None:
        return jsonify({"ok": False, "msg": "No exported CSV yet. Click 'Export CSV for Dataset' first."})

    data = request.get_json(force=True)
    try:
        idx = int(data.get("index"))
        x_blue = int(data.get("x_blue"))
        x_red = int(data.get("x_red"))
    except Exception:
        return jsonify({"ok": False, "msg": "Bad payload. Expected {index, x_blue, x_red}."})

    if idx < 0 or idx >= len(DF_GRAY):
        return jsonify({"ok": False, "msg": f"Row index out of range (0..{len(DF_GRAY)-1})."})

    row_name = str(DF_GRAY.index[idx])
    px = str(DF_GRAY.iloc[idx].get("pixels", "") or "")
    roi = _parse_pixels_to_roi(px)
    if roi is None:
        return jsonify({"ok": False, "msg": f"No ROI saved for row '{row_name}'."})

    profile = _sbr_profile_from_roi(roi)
    n = int(profile.size)

    if x_blue < 5 or x_blue > n - 6:
        return jsonify({"ok": False, "msg": f"Blue index invalid. Valid range is 5..{n-6}."})
    if x_red < 5 or x_red > n - 6:
        return jsonify({"ok": False, "msg": f"Red index invalid. Valid range is 5..{n-6}."})

    try:
        med_blue = _median_window(profile, x_blue, half_window=5)
        med_red = _median_window(profile, x_red, half_window=5)
        peak_x, peak_value = _peak_between(profile, x_blue, x_red)

        # a = median segment A (red), b = median segment B (blue)
        baseline_xhalf, baseline_value = _baseline_value(
            x_b=x_blue, x_a=x_red, b=med_blue, a=med_red
        )
    except Exception as e:
        return jsonify({"ok": False, "msg": f"Computation failed: {e}"})

    # Ensure columns exist (safe if CSV was older)
    for col in (
        "x_blue",
        "median_blue",
        "x_red",
        "median_red",
        "peak_x",
        "peak_value",
        "baseline_xhalf",
        "baseline_value",
    ):
        if col not in DF_GRAY.columns:
            DF_GRAY[col] = ""

    DF_GRAY.at[row_name, "x_blue"] = int(x_blue)
    DF_GRAY.at[row_name, "median_blue"] = float(med_blue)
    DF_GRAY.at[row_name, "x_red"] = int(x_red)
    DF_GRAY.at[row_name, "median_red"] = float(med_red)
    DF_GRAY.at[row_name, "peak_x"] = int(peak_x)
    DF_GRAY.at[row_name, "peak_value"] = float(peak_value)
    DF_GRAY.at[row_name, "baseline_xhalf"] = float(baseline_xhalf)
    DF_GRAY.at[row_name, "baseline_value"] = float(baseline_value)

    DF_GRAY.to_csv(DF_CSV_PATH)

    return jsonify(
        {
            "ok": True,
            "msg": (
                f"Updated {DF_CSV_PATH.name}: row={row_name}, "
                f"blue@{x_blue} med={med_blue:.6f}, "
                f"red@{x_red} med={med_red:.6f}, "
                f"peak@{peak_x} val={peak_value:.6f}, "
                f"baseline={baseline_value:.6f} (x_half={baseline_xhalf:.3f})"
            ),
            "row": row_name,
            "x_blue": x_blue,
            "median_blue": med_blue,
            "x_red": x_red,
            "median_red": med_red,
            "peak_x": peak_x,
            "peak_value": peak_value,
            "baseline_xhalf": baseline_xhalf,
            "baseline_value": baseline_value,
        }
    )


@app.route("/payload.json")
def payload_json():
    return jsonify(payload)


# ----------------- run server -----------------
from werkzeug.serving import make_server  # noqa

class ServerThread(threading.Thread):
    def __init__(self, flask_app: Flask):
        super().__init__(daemon=True)
        self.srv = make_server("127.0.0.1", 0, flask_app)
        self.port = self.srv.server_port

    def run(self):
        self.srv.serve_forever()

server = ServerThread(app)
server.start()

try:
    from google.colab import output as colab_output  # type: ignore

    proxied = colab_output.eval_js(f"google.colab.kernel.proxyPort({server.port})")
    display(
        HTML(
            f'<a href="{proxied}" target="_blank" '
            f'style="font-weight:700;background:#2a6de0;color:#fff;'
            f'padding:10px 14px;border-radius:8px;text-decoration:none">Open UW-LFA GUI in a new tab</a>'
        )
    )
except Exception:
    print(f"Open: http://127.0.0.1:{server.port}")

print(f"Repo: {REPO_DIR}\nDatabase: {DB_DIR}\n")
print("Loaded folders:")
for k in DATASETS:
    print(f" - {k}: {[d['filename'] for d in DATASETS[k]]}")
print(f"\nSaving to: {(EXPORT_ROOT / DATE_FOLDER).as_posix()}")


Repo: /content/_uwlfa_tmp/UW-LFA-Analysis-main
Database: /content/_uwlfa_tmp/UW-LFA-Analysis-main/100microliters/Database

Loaded folders:
 - 11-1-23_n=3: ['1.5e5_n=3.jpg', '1e5_n=3.jpg', '1e7_n=3.jpg', '5e5_n=3.jpg', '5e6_n=3.jpg', 'CC_1e6_n=3.jpg', 'K_1e6_n=3.jpg', 'dip_n_3.jpg', 'neg_n_3.jpg', 'pos_n_3.jpg']
 - 8-3-23_n=1: ['1.5e5_n=1.jpg', '1e5_n=1.jpg', '1e6_n=1.jpg', '1e7_n=1.jpg', '5e5_n=1.jpg', '5e6_n=1.jpg', 'dip_n_1.jpg', 'neg_ctrl_n_1.jpg', 'pos_ctrl_n_1.jpg']
 - 9-6-23_n=2: ['1.5e5_n=2.jpg', '1e5_n=2.jpg', '1e6_n=2.jpg', '1e7_n=2.jpg', '5e5_n=2.jpg', '5e6_n=2.jpg', 'dips_n_2.jpg', 'neg_ctrls_n_2.jpg', 'pos_ctrl_n_2.jpg']
 - LossLessFormat_N1: ['1.5e5_n=1.tiff', '1e5_n=1.tiff', '1e6_n=1.tiff', '1e7_n=1.tiff', '5e5_n=1.tiff', '5e6_n=1.tiff', 'neg_ctrl_n_1.tiff', 'pos_ctrl_n_1.tiff']
 - LossLessFormat_N2: ['1.5e5_n=2.tiff', '1e5_n=2.tiff', '1e6_n=2.tiff', '1e7_n=2.tiff', '5e5_n=2.tiff', '5e6_n=2.tiff', 'neg_ctrls_n_2.tiff', 'pos_ctrl_n_2.tiff']
 - LossLessFormat_N3: ['1.5e5_n=

In [ ]:
import shutil
from pathlib import Path

# Define the root export directory and the specific date folder
EXPORT_ROOT = Path("/content/roi_exports")
DATE_FOLDER = "2026-01-26"  # This should dynamically match the DATE_FOLDER from the first cell

folder_to_delete = EXPORT_ROOT / DATE_FOLDER

if folder_to_delete.exists():
    print(f"Deleting folder: {folder_to_delete.as_posix()}")
    shutil.rmtree(folder_to_delete)
    print("Folder deleted successfully.")
else:
    print(f"Folder not found: {folder_to_delete.as_posix()}")


Deleting folder: /content/roi_exports/2026-01-26
Folder deleted successfully.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
Mounted at /content/drive
